In [23]:
import pandas as pd
import re

# pandas 3.0 uses new Arrow string type by default which crashes with numpy 2.4
# this setting reverts to old object dtype which is compatible
pd.options.future.infer_string = False

df = pd.read_csv('../data/raw/agents.csv')
print("Shape:", df.shape)
df.head()

Shape: (312, 10)


,agent_id,agency_id,first_name,last_name,email,phone,license_number,hire_date,is_active,commission_rate
0,AGT-0001,AGY-0010,Joseph,Moody,ronaldstephens@example.net,(642)910-2499,AG-299122,2019-06-28,no,0.0319
1,AGT-0002,AGY-0048,David,Barker,gbender@example.net,271.690.6594x0139,AG-428838,2011-04-25,True,0.0223
2,AGT-0003,AGY-0033,Barbara,Hines,thomaspearson@example.com,(796)371-7565,AG-994141,2023-02-07,no,0.0412
3,AGT-0004,AGY-0012,Mia,Rivera,zroberts@example.net,2719545168,AG-171849,2011-04-11,False,0.0438
4,AGT-0005,AGY-0008,Robert,Lopez,cindy77@example.org,548.424.7710,AG-697347,2015-04-11,yes,0.0298


In [24]:
# Step 1: Remove Duplicate Rows

# check how many duplicates exist before cleaning
print("Duplicates found:", df.duplicated().sum())
print("Before:", df.shape[0], "rows")

# drop duplicate rows, keep first occurrence
df = df.drop_duplicates().reset_index(drop=True)

print("After: ", df.shape[0], "rows")
print("Removed:", 312 - df.shape[0], "duplicate rows")

Duplicates found: 12
Before: 312 rows
After:  300 rows
Removed: 12 duplicate rows


In [25]:
# Step 2: Standardize is_active Column
# problem: column has 6 different formats — 'yes', 'no', 'True', 'False', '1', '0'
# solution: map all variations to True/False boolean

print("Before - unique values:", df['is_active'].unique())

# define which values mean True and which mean False
true_values  = ['yes', 'True', '1']
false_values = ['no', 'False', '0']

df['is_active'] = df['is_active'].map(lambda x: True if x in true_values else False)

print("After  - unique values:", df['is_active'].unique())
print("Data type:", df['is_active'].dtype)

Before - unique values: ['no' 'True' 'False' 'yes' '0' '1']
After  - unique values: [False  True]
Data type: bool


In [26]:
# Step 3: Convert hire_date to DateTime
# problem: hire_date is stored as string (object), should be datetime for date operations

print("Before dtype:", df['hire_date'].dtype)
print("Sample:", df['hire_date'].head(3).tolist())

# convert string to datetime — format is already consistent (YYYY-MM-DD)
df['hire_date'] = pd.to_datetime(df['hire_date'])

print("After dtype: ", df['hire_date'].dtype)
print("Sample:", df['hire_date'].head(3).tolist())

Before dtype: object
Sample: ['2019-06-28', '2011-04-25', '2023-02-07']
After dtype:  datetime64[ns]
Sample: [Timestamp('2019-06-28 00:00:00'), Timestamp('2011-04-25 00:00:00'), Timestamp('2023-02-07 00:00:00')]


In [27]:
# Step 4: Standardize Phone Numbers
# problem: multiple formats exist — (xxx)xxx-xxxx, xxx.xxx.xxxx, digits only, with extensions
# solution: remove all non-digit characters only (brackets, dots, dashes, spaces)

print("Before - sample values:", df['phone'].head(5).tolist())

# remove all non-digit characters — do NOT trim to 10 digits
# because dataset has numbers ranging from 10 to 18 digits
df['phone'] = df['phone'].apply(lambda x: re.sub(r'\D', '', str(x)))

print("After  - sample values:", df['phone'].head(5).tolist())
print("\nDigit count distribution:")
print(df['phone'].str.len().value_counts().sort_index())

Before - sample values: ['(642)910-2499', '271.690.6594x0139', '(796)371-7565', '2719545168', '548.424.7710']
After  - sample values: ['6429102499', '27169065940139', '7963717565', '2719545168', '5484247710']

Digit count distribution:
phone
10    103
11     13
13     44
14     42
15     52
16     24
17     14
18      8
Name: count, dtype: int64


In [28]:
# Step 5: Handle Invalid Emails
# problem: 9 emails don't contain '@' — these are invalid
# solution: fill invalid emails with "invalid email" instead of dropping rows

# find all rows where email does not contain '@'
invalid_mask = ~df['email'].str.contains('@', na=False)
print("Invalid emails found:", invalid_mask.sum())
print("Before:", df.loc[invalid_mask, 'email'].tolist())

# replace invalid emails with "invalid email" — row is preserved, only email is marked
df.loc[invalid_mask, 'email'] = 'invalid email'

print("After:", df.loc[invalid_mask, 'email'].tolist())
print("Total rows preserved:", df.shape[0])

Invalid emails found: 9
Before: ['cbrennanexample.net', 'amanda98example.org', 'richard85example.org', 'gallagherstacieexample.net', 'gary15example.c', 'wbarajasexample.net', 'brianscottexample.org', 'brittanybryantexample.net', 'meyerdonnaexample.c']
After: ['invalid email', 'invalid email', 'invalid email', 'invalid email', 'invalid email', 'invalid email', 'invalid email', 'invalid email', 'invalid email']
Total rows preserved: 300


In [29]:
# Step 6: Validate commission_rate Range
# expected range: 0.02 to 0.06 (2% to 6%)

print("Min :", df['commission_rate'].min())
print("Max :", df['commission_rate'].max())
print("Mean:", round(df['commission_rate'].mean(), 4))

# check if any values fall outside the expected range
out_of_range = df[(df['commission_rate'] < 0.02) | (df['commission_rate'] > 0.06)]
print("Out of range values:", len(out_of_range))

if len(out_of_range) == 0:
    print("All commission_rate values are within valid range (0.02 - 0.06)")

Min : 0.0203
Max : 0.06
Mean: 0.0403
Out of range values: 0
All commission_rate values are within valid range (0.02 - 0.06)


In [30]:
# Save Cleaned Data

df.to_csv('../data/cleaned/clean-agents.csv', index=False)

print("Cleaned file saved to: data/cleaned/clean-agents.csv")
print("Final shape:", df.shape)
print("\nFinal dtypes:")
print(df.dtypes)

Cleaned file saved to: data/cleaned/clean-agents.csv
Final shape: (300, 10)

Final dtypes:
agent_id                   object
agency_id                  object
first_name                 object
last_name                  object
email                      object
phone                      object
license_number             object
hire_date          datetime64[ns]
is_active                    bool
commission_rate           float64
dtype: object
